In [1]:
import pandas as pd
import sys

sys.path.insert(0, "..")

from src.helpers import (
    drop_and_dedup, 
    build_column_order, 
    scale_continuous
)

In [2]:
source = "../data/processed/asthma_cleaned.csv"

In [3]:
df = pd.read_csv(source)
df

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,PhysicalActivity,DietQuality,SleepQuality,...,LungFunctionFEV1,LungFunctionFVC,Wheezing,ShortnessOfBreath,ChestTightness,Coughing,NighttimeSymptoms,ExerciseInduced,Diagnosis,DoctorInCharge
0,5034,63,0,1,0,15.848744,0,0.894448,5.488696,8.701003,...,1.369051,4.941206,0,0,1,0,0,1,0,Dr_Confid
1,5035,26,1,2,2,22.757042,0,5.897329,6.341014,5.153966,...,2.197767,1.702393,1,0,0,1,1,1,0,Dr_Confid
2,5036,57,0,2,1,18.395396,0,6.739367,9.196237,6.840647,...,1.698011,5.022553,1,1,1,0,1,1,0,Dr_Confid
3,5037,40,1,2,1,38.515278,0,1.404503,5.826532,4.253036,...,3.032037,2.300159,1,0,1,1,1,0,0,Dr_Confid
4,5038,61,0,0,3,19.283802,0,4.604493,3.127048,9.625799,...,3.470589,3.067944,1,1,1,0,0,1,0,Dr_Confid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2387,7421,43,1,0,2,29.059613,0,3.019854,6.119637,8.300960,...,3.125249,5.166032,0,1,0,0,0,1,1,Dr_Confid
2388,7422,18,1,0,1,20.740850,0,5.805180,4.386992,7.731192,...,1.132977,5.509502,0,0,0,1,1,0,1,Dr_Confid
2389,7423,54,0,3,2,37.079560,0,4.735169,8.214064,7.483521,...,1.685962,3.346877,1,0,1,1,0,1,1,Dr_Confid
2390,7424,46,1,0,2,23.444712,0,9.672637,7.362861,6.717272,...,3.481549,1.713274,0,1,1,0,1,1,0,Dr_Confid


In [4]:
df_processed = df.copy()

In [5]:
TARGET = "Diagnosis"

# drop only what's unnecessary (ID/leak/constant)
drop_cols = ["PatientID", "DoctorInCharge"]

# keep these - no scaling
small_codes = ["Gender", "Ethnicity", "EducationLevel"]       

binary_cols = [
    "Smoking","PetAllergy","FamilyHistoryAsthma","HistoryOfAllergies","Eczema","HayFever",
    "GastroesophagealReflux","Wheezing","ShortnessOfBreath","ChestTightness","Coughing",
    "NighttimeSymptoms","ExerciseInduced"
]

# scale these - true continuous features
continuous_cols = [
    "Age","BMI","PhysicalActivity","DietQuality","SleepQuality",
    "PollutionExposure","PollenExposure","DustExposure",
    "LungFunctionFEV1","LungFunctionFVC"
]

In [6]:
to_drop = [c for c in drop_cols if c in df_processed.columns]

In [7]:
df_processed, actually_dropped = drop_and_dedup(df_processed, to_drop)

In [8]:
order = build_column_order(df_processed, continuous_cols, small_codes, binary_cols, TARGET)
df_processed = df_processed[order]

In [9]:
print("Dropped:", to_drop)
print("Shape after drop/dedup:", df_processed.shape)
print("Any NaNs:", df_processed.isna().any().any())

Dropped: ['PatientID', 'DoctorInCharge']
Shape after drop/dedup: (2392, 27)
Any NaNs: False


In [10]:
df_processed

,Age,BMI,PhysicalActivity,DietQuality,SleepQuality,PollutionExposure,PollenExposure,DustExposure,LungFunctionFEV1,LungFunctionFVC,...,Eczema,HayFever,GastroesophagealReflux,Wheezing,ShortnessOfBreath,ChestTightness,Coughing,NighttimeSymptoms,ExerciseInduced,Diagnosis
0,63,15.848744,0.894448,5.488696,8.701003,7.388481,2.855578,0.974339,1.369051,4.941206,...,0,0,0,0,0,1,0,0,1,0
1,26,22.757042,5.897329,6.341014,5.153966,1.969838,7.457665,6.584631,2.197767,1.702393,...,0,0,0,1,0,0,1,1,1,0
2,57,18.395396,6.739367,9.196237,6.840647,1.460593,1.448189,5.445799,1.698011,5.022553,...,0,1,0,1,1,1,0,1,1,0
3,40,38.515278,1.404503,5.826532,4.253036,0.581905,7.571845,3.965316,3.032037,2.300159,...,0,1,0,1,0,1,1,1,0,0
4,61,19.283802,4.604493,3.127048,9.625799,0.980875,3.049807,8.260605,3.470589,3.067944,...,0,1,0,1,1,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2387,43,29.059613,3.019854,6.119637,8.300960,2.483829,7.314582,3.425445,3.125249,5.166032,...,0,0,0,0,1,0,0,0,1,1
2388,18,20.740850,5.805180,4.386992,7.731192,7.733983,2.279073,6.467701,1.132977,5.509502,...,1,0,0,0,0,0,1,1,0,1
2389,54,37.079560,4.735169,8.214064,7.483521,2.794847,3.055139,9.484013,1.685962,3.346877,...,0,1,0,1,0,1,1,0,1,1
2390,46,23.444712,9.672637,7.362861,6.717272,9.448862,7.712584,5.051405,3.481549,1.713274,...,0,0,1,0,1,1,0,1,1,0


We dropped columns which will not be key to our research - ID (identifier, leak column) and DoctorInCharge (a constant)

In [11]:
SCALE_MODE = "z"

In [12]:
df_processed, scale_params = scale_continuous(df_processed, continuous_cols, mode=SCALE_MODE)

In [13]:
print("Scaled columns:", scale_params.get("cols"))

Scaled columns: ['Age', 'BMI', 'PhysicalActivity', 'DietQuality', 'SleepQuality', 'PollutionExposure', 'PollenExposure', 'DustExposure', 'LungFunctionFEV1', 'LungFunctionFVC']


In [14]:
print("Scale mode:", scale_params.get("mode"))

Scale mode: z


In [15]:
df_processed

,Age,BMI,PhysicalActivity,DietQuality,SleepQuality,PollutionExposure,PollenExposure,DustExposure,LungFunctionFEV1,LungFunctionFVC,...,Eczema,HayFever,GastroesophagealReflux,Wheezing,ShortnessOfBreath,ChestTightness,Coughing,NighttimeSymptoms,ExerciseInduced,Diagnosis
0,0.965740,-1.582769,-1.432099,0.160113,0.971063,0.809355,-0.780866,-1.401921,-1.368934,0.920608,...,0,0,0,0,0,1,0,0,1,0
1,-0.747054,-0.623300,0.291269,0.453069,-1.076746,-1.036866,0.810184,0.560684,-0.407132,-1.564256,...,0,0,0,1,0,0,1,1,1,0
2,0.687989,-1.229074,0.581330,1.434458,-0.102976,-1.210374,-1.267434,0.162295,-0.987146,0.983019,...,0,1,0,1,1,1,0,1,1,0
3,-0.098970,1.565307,-1.256398,0.276233,-1.596880,-1.509757,0.849659,-0.355611,0.561114,-1.105641,...,0,1,0,1,0,1,1,1,0,0
4,0.873156,-1.105686,-0.154081,-0.651625,1.504976,-1.373822,-0.713717,1.146977,1.070095,-0.516586,...,0,1,0,1,1,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2387,0.039905,0.252042,-0.699950,0.376978,0.740107,-0.861740,0.760717,-0.544470,0.669296,1.093099,...,0,0,0,0,1,0,0,0,1,1
2388,-1.117388,-0.903322,0.259526,-0.218561,0.411163,0.927074,-0.980178,0.519779,-1.642920,1.356614,...,1,0,0,0,0,0,1,1,0,1
2389,0.549114,1.365905,-0.109067,1.096868,0.268175,-0.755772,-0.711873,1.574952,-1.001130,-0.302584,...,0,1,0,1,0,1,1,0,1,1
2390,0.178780,-0.527792,1.591768,0.804295,-0.174204,1.511361,0.898316,0.024327,1.082816,-1.555908,...,0,0,1,0,1,1,0,1,1,0


In [16]:
stats = df_processed[continuous_cols].agg(["mean","std","min","max"]).T.round(3)
stats

,mean,std,min,max
Age,0.0,1.0,-1.719,1.706
BMI,0.0,1.0,-1.696,1.770
PhysicalActivity,0.0,1.0,-1.740,1.703
DietQuality,-0.0,1.0,-1.725,1.711
SleepQuality,-0.0,1.0,-1.742,1.719
PollutionExposure,0.0,1.0,-1.708,1.699
PollenExposure,0.0,1.0,-1.768,1.689
DustExposure,0.0,1.0,-1.742,1.755
LungFunctionFEV1,0.0,1.0,-1.797,1.684
LungFunctionFVC,-0.0,1.0,-1.720,1.732


Our data is successfully preprocessed and cleaned now!

In [17]:
out_dir = "../data/processed"

In [20]:
df_processed.to_csv(out_dir + "/asthma_preprocessed.csv", index=False)

In [21]:
print("Saved cleaned to:", out_dir + "/asthma_preprocessed.csv")

Saved cleaned to: ../data/processed/asthma_preprocessed.csv
